<a href="https://colab.research.google.com/github/peremartra/optipfair/blob/main/examples/expansion_divisor_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OptiPFair Notebook Series – Example: expansion_divisor Parameter

![optiPfair Logo](https://github.com/peremartra/optipfair/blob/main/images/optiPfair.png?raw=true)

This notebook demonstrates how to use the `expansion_divisor` parameter in [OptiPFair](https://github.com/peremartra/optipfair) for hardware-optimized pruning of transformer models with GLU-based MLP layers.

The `expansion_divisor` parameter ensures that the intermediate layer size is divisible by specific values (32, 64, 128, or 256), which can improve performance on modern GPUs and TPUs.

## Recommended Environment

- **Platform**: [Google Colab](https://colab.research.google.com)  
- **Hardware**: GPU runtime (recommended: T4 or better for 1B–3B models)  
- **Dependencies**: Installed automatically in the first cell (optipfair, transformers, torch)

## by Pere Martra

- [LinkedIn](https://www.linkedin.com/in/pere-martra)  
- [GitHub](https://github.com/peremartra)  
- [X / Twitter](https://x.com/peremartra)

---

> If you find this useful, please ⭐ the [repository](https://github.com/peremartra/optipfair) and share it!

---

If you want your favorite LLM to create code with optiPfair, you just need to provide it with the file: [**optipfair_llm_reference_manual.txt**](https://github.com/peremartra/optipfair/blob/main/optipfair_llm_reference_manual.txt), which contains all the necessary information for the LLM to become an expert in using the library.

# expansion_divisor Parameter Example

This notebook demonstrates the use of the `expansion_divisor` parameter for hardware-optimized model pruning.

**What is expansion_divisor?**

The `expansion_divisor` parameter rounds the intermediate layer size to the nearest multiple of a specified value (32, 64, 128, or 256). This is beneficial because:

- Many GPU/TPU architectures perform better with tensor dimensions that are multiples of certain values
- Helps ensure efficient memory access patterns
- Can improve inference speed on specific hardware

**Key Points:**
- Must be used with either `pruning_percentage` or `expansion_rate` (cannot be used alone)
- Rounding occurs after the initial pruning calculation
- Rounds to the nearest multiple (up or down)

Author: Pere Martra

Designed for Google Colab - GPU runtime recommended

---
## Installation and Setup

In [1]:
!pip install -q transformers optipfair torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.4 MB/s eta 0:00:00


## Import Libraries and Check GPU

In [2]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from optipfair import prune_model

# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.7 GB


## Configuration

In [3]:
# Model configuration
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# Pruning configurations to test
PRUNING_PERCENTAGE = 20  # Base pruning percentage

# expansion_divisor values to compare
DIVISOR_VALUES = [None, 32, 256]  # None, small divisor, large divisor

print("Configuration set successfully!")
print(f"Model: {MODEL_NAME}")
print(f"Pruning percentage: {PRUNING_PERCENTAGE}%")
print(f"Testing divisor values: {DIVISOR_VALUES}")

Configuration set successfully!
Model: meta-llama/Llama-3.2-1B
Pruning percentage: 20%
Testing divisor values: [None, 32, 256]


## Utility Functions

In [4]:
def count_parameters(model):
    """Count total parameters in model"""
    return sum(p.numel() for p in model.parameters())

def get_intermediate_size(model):
    """Get the intermediate size from the first MLP layer"""
    # Try to access model layers (works for LLaMA-style models)
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        return model.model.layers[0].mlp.gate_proj.out_features
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'h'):
        # For GPT-2 style models
        return model.transformer.h[0].mlp.c_fc.out_features
    else:
        return None

def print_model_structure(model, label="Model"):
    """Print detailed model structure information"""
    param_count = count_parameters(model)
    intermediate_size = get_intermediate_size(model)
    hidden_size = model.config.hidden_size if hasattr(model, 'config') else None

    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    print(f"Total parameters: {param_count:,}")
    if intermediate_size:
        print(f"Intermediate size: {intermediate_size:,}")
        if hidden_size:
            expansion_rate = (intermediate_size / hidden_size) * 100
            print(f"Hidden size: {hidden_size:,}")
            print(f"Expansion rate: {expansion_rate:.2f}%")

            # Check divisibility
            for divisor in [32, 64, 128, 256]:
                is_divisible = (intermediate_size % divisor) == 0
                symbol = "✓" if is_divisible else "✗"
                print(f"  {symbol} Divisible by {divisor}: {is_divisible}")
    print(f"{'='*60}\n")

    return param_count, intermediate_size

def cleanup_memory():
    """Clean up GPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Utility functions defined successfully!")

Utility functions defined successfully!


## Load Base Model

Let's load the base model and examine its structure before pruning.

In [ ]:
print(f"Loading model: {MODEL_NAME}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set pad token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [6]:
# Print original model structure
original_params, original_intermediate = print_model_structure(model, "Original Model")


Original Model
Total parameters: 1,235,814,400
Intermediate size: 8,192
Hidden size: 2,048
Expansion rate: 400.00%
  ✓ Divisible by 32: True
  ✓ Divisible by 64: True
  ✓ Divisible by 128: True
  ✓ Divisible by 256: True



## Example 1: Pruning with Different expansion_divisor Values

Let's apply the same pruning percentage (20%) with different `expansion_divisor` values and compare the results.

In [7]:
results = []

for divisor in DIVISOR_VALUES:
    print(f"\n{'#'*60}")
    print(f"Testing with expansion_divisor={divisor}")
    print(f"{'#'*60}")

    # Reload model for each test
    cleanup_memory()
    test_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
        device_map="auto"
    )

    # Apply pruning
    pruned_model, stats = prune_model(
        model=test_model,
        pruning_type="MLP_GLU",
        neuron_selection_method="MAW",
        pruning_percentage=PRUNING_PERCENTAGE,
        expansion_divisor=divisor,
        show_progress=True,
        return_stats=True
    )

    # Print results
    pruned_params, pruned_intermediate = print_model_structure(
        pruned_model,
        f"Pruned Model (divisor={divisor})"
    )

    # Store results
    results.append({
        'divisor': divisor,
        'original_params': stats['original_parameters'],
        'pruned_params': stats['pruned_parameters'],
        'reduction': stats['reduction'],
        'percentage_reduction': stats['percentage_reduction'],
        'expansion_rate': stats['expansion_rate'],
        'intermediate_size': pruned_intermediate
    })

    # Clean up
    del test_model, pruned_model
    cleanup_memory()

print("\n" + "="*60)
print("COMPARISON COMPLETE")
print("="*60)


############################################################
Testing with expansion_divisor=None
############################################################


Pruning layers: 100%|██████████| 16/16 [00:06<00:00,  2.39it/s]



Pruned Model (divisor=None)
Total parameters: 1,074,792,448
Intermediate size: 6,554
Hidden size: 2,048
Expansion rate: 320.02%
  ✗ Divisible by 32: False
  ✗ Divisible by 64: False
  ✗ Divisible by 128: False
  ✗ Divisible by 256: False


############################################################
Testing with expansion_divisor=32
############################################################


Pruning layers: 100%|██████████| 16/16 [00:04<00:00,  3.23it/s]



Pruned Model (divisor=32)
Total parameters: 1,075,382,272
Intermediate size: 6,560
Hidden size: 2,048
Expansion rate: 320.31%
  ✓ Divisible by 32: True
  ✗ Divisible by 64: False
  ✗ Divisible by 128: False
  ✗ Divisible by 256: False


############################################################
Testing with expansion_divisor=256
############################################################


Pruning layers: 100%|██████████| 16/16 [00:05<00:00,  2.72it/s]



Pruned Model (divisor=256)
Total parameters: 1,084,819,456
Intermediate size: 6,656
Hidden size: 2,048
Expansion rate: 325.00%
  ✓ Divisible by 32: True
  ✓ Divisible by 64: True
  ✓ Divisible by 128: True
  ✓ Divisible by 256: True


COMPARISON COMPLETE


## Results Comparison

Let's compare all the results side by side.

In [8]:
import pandas as pd

# Create comparison table
df = pd.DataFrame(results)
df['divisor'] = df['divisor'].fillna('None').astype(str)

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)
print(f"\n{df.to_string(index=False)}\n")

# Calculate differences
print("\nKEY OBSERVATIONS:")
print("-" * 80)

# Compare intermediate sizes
base_result = results[0]  # divisor=None
print(f"\nIntermediate Size Comparison (base: {base_result['intermediate_size']})")
for result in results:
    if result['divisor'] != base_result['divisor']:
        diff = result['intermediate_size'] - base_result['intermediate_size']
        print(f"  divisor={result['divisor']:>3}: {result['intermediate_size']} "
              f"(difference: {diff:+d})")

# Compare parameter reductions
print(f"\nParameter Reduction Comparison")
for result in results:
    print(f"  divisor={str(result['divisor']):>4}: {result['reduction']:>10,} parameters "
          f"({result['percentage_reduction']:.2f}%)")

# Compare expansion rates
print(f"\nFinal Expansion Rate Comparison")
for result in results:
    print(f"  divisor={str(result['divisor']):>4}: {result['expansion_rate']:.2f}%")

print("\n" + "="*80)


RESULTS SUMMARY

divisor  original_params  pruned_params  reduction  percentage_reduction  expansion_rate  intermediate_size
   None       1235814400     1074792448  161021952             13.029623      320.019531               6554
   32.0       1235814400     1075382272  160432128             12.981895      320.312500               6560
  256.0       1235814400     1084819456  150994944             12.218254      325.000000               6656


KEY OBSERVATIONS:
--------------------------------------------------------------------------------

Intermediate Size Comparison (base: 6554)
  divisor= 32: 6560 (difference: +6)
  divisor=256: 6656 (difference: +102)

Parameter Reduction Comparison
  divisor=None: 161,021,952 parameters (13.03%)
  divisor=  32: 160,432,128 parameters (12.98%)
  divisor= 256: 150,994,944 parameters (12.22%)

Final Expansion Rate Comparison
  divisor=None: 320.02%
  divisor=  32: 320.31%
  divisor= 256: 325.00%



## Example 2: Using expansion_divisor with expansion_rate

The `expansion_divisor` parameter also works with `expansion_rate` instead of `pruning_percentage`.

In [12]:
print("\n" + "="*60)
print("Example 2: expansion_divisor with expansion_rate")
print("="*60)

TARGET_EXPANSION_RATE = 200  # Target 200% expansion rate

# Load fresh model
cleanup_memory()
model_expansion = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto"
)

print(f"\nTarget expansion rate: {TARGET_EXPANSION_RATE}%")
print(f"Using expansion_divisor: 128")

# Apply pruning
pruned_expansion, stats_expansion = prune_model(
    model=model_expansion,
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW",
    pruning_percentage=None,
    expansion_rate=TARGET_EXPANSION_RATE,
    expansion_divisor=128,
    show_progress=True,
    return_stats=True
)

# Print results
print_model_structure(pruned_expansion, "Pruned Model (expansion_rate + divisor)")

print(f"\nTarget expansion rate: {TARGET_EXPANSION_RATE}%")
print(f"Actual expansion rate: {stats_expansion['expansion_rate']:.2f}%")
print(f"Parameter reduction: {stats_expansion['percentage_reduction']:.2f}%")

# Clean up
del model_expansion, pruned_expansion
cleanup_memory()


Example 2: expansion_divisor with expansion_rate

Target expansion rate: 200%
Using expansion_divisor: 128


Pruning layers: 100%|██████████| 16/16 [00:07<00:00,  2.18it/s]



Pruned Model (expansion_rate + divisor)
Total parameters: 833,161,216
Intermediate size: 4,096
Hidden size: 2,048
Expansion rate: 200.00%
  ✓ Divisible by 32: True
  ✓ Divisible by 64: True
  ✓ Divisible by 128: True
  ✓ Divisible by 256: True


Target expansion rate: 200%
Actual expansion rate: 200.00%
Parameter reduction: 32.58%


## Example 3: Comparing Model Structure Before and After

Let's do a detailed comparison of the model structure before and after pruning with `expansion_divisor`.

In [13]:
print("\n" + "="*60)
print("Example 3: Detailed Structure Comparison")
print("="*60)

# Load models for comparison
cleanup_memory()

print("\n--- Loading original model ---")
model_original = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto"
)

print("\n--- Pruning with expansion_divisor=128 ---")
model_pruned = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto"
)

pruned_result, stats_comparison = prune_model(
    model=model_pruned,
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW",
    pruning_percentage=20,
    expansion_divisor=128,
    show_progress=True,
    return_stats=True
)

# Get layer details
def get_layer_details(model, layer_idx=0):
    """Get details of a specific layer's MLP"""
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        layer = model.model.layers[layer_idx]
        return {
            'gate_proj_out': layer.mlp.gate_proj.out_features,
            'up_proj_out': layer.mlp.up_proj.out_features,
            'down_proj_in': layer.mlp.down_proj.in_features,
            'hidden_size': layer.mlp.gate_proj.in_features
        }
    return None

original_details = get_layer_details(model_original)
pruned_details = get_layer_details(pruned_result)

print("\n" + "="*80)
print("LAYER 0 MLP STRUCTURE COMPARISON")
print("="*80)
print(f"\n{'Component':<20} {'Original':<15} {'Pruned':<15} {'Change':<15}")
print("-" * 80)

if original_details and pruned_details:
    print(f"{'Hidden size':<20} {original_details['hidden_size']:<15,} "
          f"{pruned_details['hidden_size']:<15,} {'(unchanged)':<15}")

    for key in ['gate_proj_out', 'up_proj_out', 'down_proj_in']:
        orig = original_details[key]
        pruned = pruned_details[key]
        change = pruned - orig
        change_pct = (change / orig) * 100

        label = key.replace('_', ' ').title()
        print(f"{label:<20} {orig:<15,} {pruned:<15,} "
              f"{change:+,} ({change_pct:+.1f}%)")

print("\n" + "="*80)

# Clean up
del model_original, model_pruned, pruned_result
cleanup_memory()


Example 3: Detailed Structure Comparison

--- Loading original model ---

--- Pruning with expansion_divisor=128 ---


Pruning layers: 100%|██████████| 16/16 [00:05<00:00,  2.77it/s]



LAYER 0 MLP STRUCTURE COMPARISON

Component            Original        Pruned          Change         
--------------------------------------------------------------------------------
Hidden size          2,048           2,048           (unchanged)    
Gate Proj Out        8,192           6,528           -1,664 (-20.3%)
Up Proj Out          8,192           6,528           -1,664 (-20.3%)
Down Proj In         8,192           6,528           -1,664 (-20.3%)



## Key Takeaways

### When to Use expansion_divisor

1. **Hardware Optimization**: Use when targeting specific GPU/TPU architectures
   - NVIDIA GPUs often perform best with multiples of 128 or 256
   - Some TPUs prefer multiples of 64

2. **Deployment Constraints**: Use when you need exact dimension requirements
   - Some inference frameworks have specific alignment requirements
   - Ensures compatibility with quantization libraries

3. **Batch Processing**: Can improve performance in high-throughput scenarios
   - Better memory alignment for batched operations
   - More efficient use of tensor cores

### Best Practices

- **Start with None**: First try pruning without `expansion_divisor` to see base results
- **Common values**: 128 and 256 are good defaults for modern GPUs
- **Test performance**: Always benchmark on your target hardware
- **Consider trade-offs**: Rounding may slightly increase or decrease parameter count

### Important Notes

- `expansion_divisor` cannot be used alone - requires `pruning_percentage` or `expansion_rate`
- Valid values: `None`, `32`, `64`, `128`, `256`
- Rounding occurs after the initial pruning calculation
- The final size is always kept within valid bounds (≥1 and ≤ original size)

---
## ✅ Success! What's Next?

Congratulations! You've learned how to use the `expansion_divisor` parameter for hardware-optimized model pruning with OptiPFair.

If you found this notebook useful, the best way to support the OptiPFair project is by **starring it on GitHub**. Your support helps boost the project's visibility and reach more developers and researchers.

### ➡️ [**Star OptiPFair on GitHub**](https://github.com/peremartra/optipfair)

---

You can also follow my work and new projects on:

* **[LinkedIn](https://www.linkedin.com/in/pere-martra/)**
* **[X / Twitter](https://twitter.com/PereMartra)**